Formataçãodo do dados do DataFrame de Infodengue 

In [1]:
import pandas as pd

# Base escolhida: InfoDengue
# Cidade analisada: Fortaleza
# Período da consulta: 2017 até 2024

geocode = 2304400
doenca = "dengue"
ano_inicial = 2017
ano_final = 2025
semana_inicial = 1
semana_final = 53

arquivo_csv = "dados_infodengue/infodengue_fortaleza_2017_2024.csv"
arquivo_relatorio = "dados_formatados/INFODENGUE_DADOS_FORTALEZA.CSV"

# montando a url da API
url = (
    "https://info.dengue.mat.br/api/alertcity?"
    f"geocode={geocode}"
    f"&disease={doenca}"
    f"&format=csv"
    f"&ew_start={semana_inicial}"
    f"&ew_end={semana_final}"
    f"&ey_start={ano_inicial}"
    f"&ey_end={ano_final}"
)

print("URL usada:")
print(url)

# leitura da base
df = pd.read_csv(url)

print("\nBase carregada com sucesso")
print("Dimensão da base:", df.shape)

print("\nPrimeiras linhas:")
print(df.head())

print("\nColunas encontradas:")
print(df.columns.tolist())

# convertendo a coluna de data
df["data_iniSE"] = pd.to_datetime(df["data_iniSE"], errors="coerce")

# salvando o csv baixado
df.to_csv(arquivo_csv, index=False, encoding="utf-8")
print(f"\nCSV salvo com sucesso: {arquivo_csv}")

URL usada:
https://info.dengue.mat.br/api/alertcity?geocode=2304400&disease=dengue&format=csv&ew_start=1&ew_end=53&ey_start=2017&ey_end=2025

Base carregada com sucesso
Dimensão da base: (470, 31)

Primeiras linhas:
   data_iniSE      SE  casos_est  casos_est_min  casos_est_max  casos  \
0  2025-12-28  202553       54.0             54             54     54   
1  2025-12-21  202552       46.0             46             46     46   
2  2025-12-14  202551       51.0             51             51     51   
3  2025-12-07  202550       75.0             75             75     75   
4  2025-11-30  202549       75.0             75             75     75   

      p_rt1  p_inc100k  Localidade_id  nivel  ...    umidmed    umidmin  \
0  0.395043   2.079998              0      1  ...  72.892157  53.921714   
1  0.031540   1.771850              0      1  ...  72.963371  56.631857   
2  0.103829   1.964442              0      1  ...  71.779086  54.856629   
3  0.957840   2.888886              0      1 

Início do processo de validação dos dados, selecionando as colunas relevantes para análise

In [2]:
#Pegando as colunas de interesse para o relatório
colunas_interesse = ["data_iniSE", "casos_est"]

# Gerando dataframe com as colunas de interesse
df_relatorio = df[colunas_interesse]

#alterando o nome das colunas para o relatório
df_relatorio.rename(columns={"data_iniSE": "data", "casos_est": "casos_estimados"}, inplace=True)
    
print(df_relatorio.head())

        data  casos_estimados
0 2025-12-28             54.0
1 2025-12-21             46.0
2 2025-12-14             51.0
3 2025-12-07             75.0
4 2025-11-30             75.0


Soma de caso obtidos por mês 

In [3]:
#Agrupando por mês e somando os casos estimados
#O 'MS' significa Month Start (Início do Mês)
df_mensal = df_relatorio.set_index("data").resample("MS")['casos_estimados'].sum().reset_index()

#renomeando a coluna de casos estimados para casos_mensais e data para mes_ano
df_mensal.rename(columns={"casos_estimados": "casos_mensais", "data": "mes_ano"}, inplace=True)

#readicionando a coluna de nível de alerta, pegando o valor mais frequente do nível de alerta para cada mês


print(df_mensal.head())

     mes_ano  casos_mensais
0 2017-01-01         3328.0
1 2017-02-01         5050.0
2 2017-03-01         8238.0
3 2017-04-01        14309.0
4 2017-05-01         3780.0


Adicionando uma coluna de nivel de alerta

In [4]:
import numpy as np

#Calcular a média e o desvio padrão histórico de fortaleza
media_casos  = df_mensal["casos_mensais"].mean()
desvio_padrao = df_mensal["casos_mensais"].std()

#Defenir a lógica de alerta
def definir_alerta(casos):
    if casos <= media_casos:
        return 1 # Alerta baixo
    elif casos <= (media_casos + desvio_padrao):
        return 2 # Alerta médio
    elif casos <= (media_casos + 2 * desvio_padrao):
        return 3 # Alerta alto
    else:
        return 4 # Alerta crítico
    

# Criando a nova coluna de nível de alerta
df_mensal["nivel_alerta"] = df_mensal['casos_mensais'].apply(definir_alerta) 

print(df_mensal.head())

     mes_ano  casos_mensais  nivel_alerta
0 2017-01-01         3328.0             2
1 2017-02-01         5050.0             3
2 2017-03-01         8238.0             4
3 2017-04-01        14309.0             4
4 2017-05-01         3780.0             2


Exporta o Dataframe

In [5]:
#exportando o dataframe para um arquivo csv
df_mensal.to_csv(arquivo_relatorio, index=False, encoding="utf-8")
print(f"\nRelatório salvo com sucesso: {arquivo_relatorio}")


Relatório salvo com sucesso: dados_formatados/INFODENGUE_DADOS_FORTALEZA.CSV
